In [ ]:
!pip install -q scgpt scanpy gdown langgraph


In [ ]:
import os
from typing import Any, TypedDict

import gdown
import numpy as np
import pandas as pd
import scanpy as sc
import torch
import torch.nn as nn
import torch.nn.functional as F
from langgraph.graph import END, StateGraph
from sklearn.model_selection import train_test_split

DATASET_FOLDER_URL = "https://drive.google.com/drive/folders/1OjwOq4DtShEci0jBgF7FpXsH07l21k4k"
MODEL_FOLDER_URL = "https://drive.google.com/drive/folders/1oWh_-ZRdhtoGQ2Fw24HP41FgLoomVo-y"

DATA_DIR = "/content/yan_dataset"
MODEL_DIR = "/content/scgpt_ckpt"
CONFIDENCE_THRESHOLD = 0.80


In [ ]:
def patch_torchtext_compat() -> None:
    import sys
    from types import ModuleType

    for key in list(sys.modules.keys()):
        if "scgpt" in key or "torchtext" in key:
            del sys.modules[key]

    class _Vocab:
        def __init__(self, vocab=None):
            self._stoi = dict(vocab) if isinstance(vocab, dict) else {}
            self._itos = list(self._stoi.keys())
            self._default_index = None

        def __len__(self):
            return len(self._stoi)

        def __contains__(self, token):
            return token in self._stoi

        def get_stoi(self):
            return self._stoi

        def get_itos(self):
            return self._itos

        def __getitem__(self, token):
            return self._stoi.get(token, self._default_index if self._default_index is not None else 0)

        def __call__(self, tokens):
            if isinstance(tokens, list):
                return [self._stoi.get(t, self._default_index or 0) for t in tokens]
            return self._stoi.get(tokens, self._default_index or 0)

        def set_default_index(self, idx):
            self._default_index = idx

        def get_default_index(self):
            return self._default_index

        def append_token(self, token):
            if token not in self._stoi:
                self._stoi[token] = len(self._itos)
                self._itos.append(token)

        def insert_token(self, token, index):
            if token not in self._stoi:
                self._itos.insert(index, token)
                self._stoi = {t: i for i, t in enumerate(self._itos)}

    class _VocabBuilt(_Vocab):
        def __init__(self, ordered_dict, min_freq=1):
            super().__init__()
            for token, freq in ordered_dict.items():
                if freq >= min_freq:
                    self._stoi[token] = len(self._itos)
                    self._itos.append(token)
            self.vocab = self._stoi

    def _vocab_builder(ordered_dict, min_freq=1, **kwargs):
        return _VocabBuilt(ordered_dict, min_freq)

    _torchtext = ModuleType("torchtext")
    _torchtext_vocab = ModuleType("torchtext.vocab")
    _torchtext_vocab.Vocab = _Vocab
    _torchtext_vocab.vocab = _vocab_builder
    _torchtext.vocab = _torchtext_vocab

    sys.modules["torchtext"] = _torchtext
    sys.modules["torchtext.vocab"] = _torchtext_vocab


class PipelineState(TypedDict, total=False):
    dataset_dir: str
    model_dir: str
    threshold: float
    adata: Any
    embeddings: np.ndarray
    labels: np.ndarray
    pred_labels: list[str]
    confidence_scores: np.ndarray
    confidence_gate: list[str]
    high_conf_count: int
    low_conf_count: int


class CellTypeMLP(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, num_classes: int) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


def _download_folder_if_missing(url: str, output_dir: str) -> None:
    os.makedirs(output_dir, exist_ok=True)
    if os.listdir(output_dir):
        return
    gdown.download_folder(url=url, output=output_dir, quiet=False, use_cookies=False)


def raw_dataset_node(state: PipelineState) -> PipelineState:
    dataset_dir = state.get("dataset_dir", DATA_DIR)
    model_dir = state.get("model_dir", MODEL_DIR)
    _download_folder_if_missing(DATASET_FOLDER_URL, dataset_dir)
    _download_folder_if_missing(MODEL_FOLDER_URL, model_dir)
    return {"dataset_dir": dataset_dir, "model_dir": model_dir}


def dataset_processing_node(state: PipelineState) -> PipelineState:
    dataset_dir = state["dataset_dir"]
    data = pd.read_csv(os.path.join(dataset_dir, "yan_data.csv"))
    cell = pd.read_csv(os.path.join(dataset_dir, "yan_celldata.csv"))

    data.index = data.iloc[:, 0]
    data = data.iloc[:, 1:]

    x_df = data.T.reset_index().rename(columns={"index": "cell_id"})
    df = x_df.merge(cell, left_on="cell_id", right_on="Unnamed: 0").drop(columns=["Unnamed: 0"])

    x = df.drop(columns=["cell_id", "cell_type1", "cell_type2"])
    y = df["cell_type1"]

    adata = sc.AnnData(x)
    adata.var_names = x.columns.astype(str)
    adata.obs["cell_type"] = y.values
    return {"adata": adata}


def scgpt_node(state: PipelineState) -> PipelineState:
    patch_torchtext_compat()
    from scgpt.tasks import embed_data

    adata = state["adata"]
    model_dir = state["model_dir"]
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    adata = embed_data(
        adata,
        model_dir=model_dir,
        gene_col="index",
        max_length=1200,
        batch_size=64,
        device=device,
        use_fast_transformer=False,
        return_new_adata=False,
    )

    embeddings = np.asarray(adata.obsm["X_scGPT"], dtype=np.float32)
    labels = adata.obs["cell_type"].values
    return {"adata": adata, "embeddings": embeddings, "labels": labels}


def prediction_node(state: PipelineState) -> PipelineState:
    embeddings = state["embeddings"]
    labels = state["labels"]
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    classes = sorted(np.unique(labels))
    label_to_idx = {c: i for i, c in enumerate(classes)}
    idx_to_label = {i: c for c, i in label_to_idx.items()}

    x_np = np.asarray(embeddings, dtype=np.float32)
    y_np = np.asarray([label_to_idx[l] for l in labels], dtype=np.int64)

    x_train_np, x_test_np, y_train_np, _ = train_test_split(
        x_np,
        y_np,
        test_size=0.2,
        random_state=42,
        stratify=y_np,
    )

    x_train = torch.tensor(x_train_np, dtype=torch.float32, device=device)
    y_train = torch.tensor(y_train_np, dtype=torch.long, device=device)
    x_test = torch.tensor(x_test_np, dtype=torch.float32, device=device)

    model = CellTypeMLP(input_dim=x_train.shape[1], hidden_dim=256, num_classes=len(classes)).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    model.train()
    for _ in range(200):
        optimizer.zero_grad()
        logits_train = model(x_train)
        loss = F.cross_entropy(logits_train, y_train)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        logits_test = model(x_test)
        probs_test = torch.softmax(logits_test, dim=1)
        pred_idx_test = torch.argmax(probs_test, dim=1)
        confidence_test = torch.max(probs_test, dim=1).values

    pred_labels_test = [idx_to_label[int(i)] for i in pred_idx_test.cpu().numpy()]
    confidence_scores_test = confidence_test.cpu().numpy()

    for i in range(min(5, len(pred_labels_test))):
        print(i, pred_labels_test[i], float(confidence_scores_test[i]))

    return {"pred_labels": pred_labels_test, "confidence_scores": confidence_scores_test}


def confidence_check_node(state: PipelineState) -> PipelineState:
    scores = state["confidence_scores"]
    threshold = float(state.get("threshold", CONFIDENCE_THRESHOLD))

    confidence_gate = ["High" if s >= threshold else "Low" for s in scores]
    high_conf_count = sum(g == "High" for g in confidence_gate)
    low_conf_count = len(confidence_gate) - high_conf_count

    print(f"Confidence check complete: High={high_conf_count}, Low={low_conf_count}, threshold={threshold}")
    return {
        "confidence_gate": confidence_gate,
        "high_conf_count": high_conf_count,
        "low_conf_count": low_conf_count,
        "threshold": threshold,
    }


def build_graph():
    graph = StateGraph(PipelineState)
    graph.add_node("raw_dataset", raw_dataset_node)
    graph.add_node("dataset_processing", dataset_processing_node)
    graph.add_node("scgpt", scgpt_node)
    graph.add_node("prediction", prediction_node)
    graph.add_node("confidence_check", confidence_check_node)

    graph.set_entry_point("raw_dataset")
    graph.add_edge("raw_dataset", "dataset_processing")
    graph.add_edge("dataset_processing", "scgpt")
    graph.add_edge("scgpt", "prediction")
    graph.add_edge("prediction", "confidence_check")
    graph.add_edge("confidence_check", END)
    return graph.compile()


In [ ]:
app = build_graph()
final_state = app.invoke(
    {
        "dataset_dir": DATA_DIR,
        "model_dir": MODEL_DIR,
        "threshold": CONFIDENCE_THRESHOLD,
    }
)

print("Pipeline finished.")
